# Custom User-based Model
The present notebooks aims at creating a UserBased class that inherits from the Algobase class (surprise package) and that can be customized with various similarity metrics, peer groups and score aggregation functions. 

In [15]:
# reloads modules automatically before entering the execution of code
%load_ext autoreload
%autoreload 2

# standard library imports
# -- add new imports here --
from surprise import Dataset, Reader
from surprise.model_selection import train_test_split
from surprise import KNNWithMeans
import heapq
from surprise import AlgoBase, PredictionImpossible

# third parties imports
import numpy as np 
import pandas as pd
from surprise import AlgoBase
# -- add new imports here --

# local imports
from constants import Constant as C
from loaders import load_ratings
# -- add new imports here --

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


# 1. Loading Data
Prepare a dataset in order to help implementing a user-based recommender system

In [3]:
# -- load data, build trainset and anti testset --
data = load_ratings(surprise_format=True)
trainset = data.build_full_trainset()
anti_testset = trainset.build_anti_testset()

# 2. Explore Surprise's user-based algorithm
Displays user-based predictions and similarity matrix on the test dataset using the KNNWithMeans class

In [ ]:


def knn_user_based_pred(trainset, anti_testset, 
                    k=3, min_k=2, 
                    sim_name='msd', user_based=True, 
                    min_support=3, display_30=False, display_matrix=False, test_user=False):
    """
    Train a KNNWithMeans user-based model and return predictions.
    
    Parameters:
    - trainset: Surprise training set
    - anti_testset: testset with unknown ratings
    - k: number of neighbors
    - min_k: minimum number of neighbors to use weighted average
    - sim_name: similarity metric name (e.g. 'msd', 'cosine', etc.)
    - user_based: True = user-user similarity
    - min_support: minimum number of common ratings to consider similarity
    - display_30: if True, display first 30 predictions
    - display_matrix: if True, print the similarity matrix
    - test_user : if True, make an estimation  for user 11 and item 364. 

    Returns:
    - predictions (list of Prediction objects)
    """
    
    sim_options = {
        'name': sim_name,
        'user_based': user_based,
        'min_support': min_support
    }

    algo = KNNWithMeans(k=k, min_k=min_k, sim_options=sim_options)
    algo.fit(trainset)
    predictions = algo.test(anti_testset)

    if test_user : 
        single_prediction = algo.predict(uid=11, iid=364)
        print(f"Estimated rating for user 11 and item 364: {single_prediction.est:.2f}")


    if display_30:
        print(f"- - - - - - - - 30 predictions (k={k}, min_k={min_k}, min_support={min_support}) - - - - - - - -")
        for prediction in predictions[:30]:
            print(f"User: {prediction.uid} | Item: {prediction.iid} | "
                  f"True rating: {prediction.r_ui} | "
                  f"Estimated rating: {prediction.est:.2f} | "
                  f"actual_k: {prediction.details['actual_k']}")

    if display_matrix:
        print("Similarity matrix:")
        print(algo.sim)

    return predictions
predictions = knn_user_based_pred(trainset, anti_testset, test_user=True)



Computing the msd similarity matrix...
Done computing similarity matrix.
Estimated rating for user 11 and item 364: 4.25


### Playing with KNN

In [ ]:
predictions = user_based_pred(trainset, anti_testset, min_k=3, display_30=True, display_matrix=True)
predictions = user_based_pred(trainset, anti_testset, min_support=1, display_30=True, display_matrix=True)



# 3. Implement and explore a customizable user-based algorithm
Create a self-made user-based algorithm allowing to customize the similarity metric, peer group calculation and aggregation function

In [18]:


class UserBased(AlgoBase):
    def __init__(self, k=3, min_k=1, sim_options={}, **kwargs):
        # Call parent constructor
        AlgoBase.__init__(self, sim_options=sim_options, **kwargs)
        self.k = k               # Number of neighbors to consider
        self.min_k = min_k       # Minimum number of neighbors required

    def fit(self, trainset):
        # Call parent fit method
        AlgoBase.fit(self, trainset)

        # Step 1: Compute the rating matrix (user-item with NaNs for missing ratings)
        self.compute_rating_matrix()

        # Step 2: Compute the similarity matrix (user-user similarity)
        self.compute_similarity_matrix()

        # Step 3: Compute the mean rating of every user
        self.mean_ratings = []
        for uid in range(self.trainset.n_users):
            ratings = [rating for (_, rating) in self.trainset.ur[uid]]
            if ratings:
                self.mean_ratings.append(np.mean(ratings))
            else:
                self.mean_ratings.append(0.0)

        return self


    def estimate(self, u, i):
        # If user or item is unknown, raise an exception
        if not (self.trainset.knows_user(u) and self.trainset.knows_item(i)):
            raise PredictionImpossible('User and/or item is unknown.')

        # Default estimate = user's mean rating
        estimate = self.mean_ratings[u]

        neighbors = []

        # Get peer group: users who rated item i
        for v, rating_vi in self.trainset.ir[i]:
            if v == u:
                continue
            sim_uv = self.sim[u, v]
            neighbors.append((v, sim_uv, rating_vi))

        # Select top-k neighbors based on similarity
        k_neighbors = heapq.nlargest(self.k, neighbors, key=lambda x: x[1])

        num, denom, actual_k = 0.0, 0.0, 0
        for v, sim, r_vi in k_neighbors:
            if sim > 0:
                mean_v = self.mean_ratings[v]
                num += sim * (r_vi - mean_v)  # centered rating
                denom += abs(sim)
                actual_k += 1

        # If we have enough neighbors, adjust the estimate
        if actual_k >= self.min_k and denom > 0:
            estimate += num / denom

        # Return estimate (either mean_u or mean_u + peer adjustment)
        return estimate

            
            
    def compute_rating_matrix(self):
        # Get number of users (m) and items (n)
        m, n = self.trainset.n_users, self.trainset.n_items

        # Create an m × n matrix filled with NaN
        self.ratings_matrix = np.empty((m, n))
        self.ratings_matrix[:] = np.nan

         # Fill matrix with known ratings (each cell [u, i] = rating)
        for uiid in range(m):
            for item_inner_id, rating in self.trainset.ur[uiid]:
                self.ratings_matrix[uiid, item_inner_id] = rating

    def compute_similarity_matrix(self):
        m = self.trainset.n_users

        # Preallocate similarity matrix: identity matrix (1 on diagonal)
        self.sim = np.eye(m)

        # Get min_support from sim_options
        min_support = self.sim_options.get('min_support', 1)

        # Loop over all user pairs 
        for i in range(m):
            row_i = self.ratings_matrix[i]
            for j in range(i + 1, m):
                row_j = self.ratings_matrix[j]

                # Compute the support as number of common non-NaN entries
                support = np.sum(~np.isnan(row_i - row_j))

                if support >= min_support:
                    # Compute MSD similarity (1 / (1 + mean squared difference))
                    sim = 1 / (1 + np.nanmean((row_i - row_j)[~np.isnan(row_i - row_j)] ** 2))
                else:
                    sim = 0.0

                # Assign the symmetric similarity
                self.sim[i, j] = sim
                self.sim[j, i] = sim



# 4. Compare KNNWithMeans with UserBased
Try to replicate KNNWithMeans with your self-made UserBased and check that outcomes are identical

In [19]:
# -- assert that predictions are the same with different sim_options --
sim_options = {
    'name': 'msd',        # mean squared difference (MSD) similarity
    'user_based': True,   # we want User-User CF, not Item-Item CF
    'min_support': 3      # minimum of 3 common ratings to be considered neighbors
}
my_algo = UserBased(k=3, min_k=2, sim_options=sim_options)
my_algo.fit(trainset)
my_predictions = my_algo.test(anti_testset)


print("- - - - - Comparison of predictions - - - - -")
knn_predictions = knn_user_based_pred(trainset, anti_testset)

for i in range(30):
    pred_knn = knn_predictions[i]       # prediction from KNNWithMeans
    pred_my = my_predictions[i]     # prediction from your UserBased model

    diff = abs(pred_knn.est - pred_my.est)

    print(f"User: {pred_my.uid} | Item: {pred_my.iid} | "
          f"KNN estimate: {pred_knn.est:.4f} | "
          f"UserBased estimate: {pred_my.est:.4f} | "
          f"Diff: {diff:.4f}")


- - - - - Comparison of predictions - - - - -
Computing the msd similarity matrix...
Done computing similarity matrix.
User: 1 | Item: 10 | KNN estimate: 2.6248 | UserBased estimate: 2.6248 | Diff: 0.0000
User: 1 | Item: 17 | KNN estimate: 3.1901 | UserBased estimate: 3.1901 | Diff: 0.0000
User: 1 | Item: 39 | KNN estimate: 2.5295 | UserBased estimate: 2.5295 | Diff: 0.0000
User: 1 | Item: 47 | KNN estimate: 2.9796 | UserBased estimate: 2.9796 | Diff: 0.0000
User: 1 | Item: 50 | KNN estimate: 4.0432 | UserBased estimate: 4.0432 | Diff: 0.0000
User: 1 | Item: 52 | KNN estimate: 2.1271 | UserBased estimate: 2.1271 | Diff: 0.0000
User: 1 | Item: 62 | KNN estimate: 2.7740 | UserBased estimate: 2.7740 | Diff: 0.0000
User: 1 | Item: 110 | KNN estimate: 2.6954 | UserBased estimate: 2.6954 | Diff: 0.0000
User: 1 | Item: 144 | KNN estimate: 2.1305 | UserBased estimate: 2.1305 | Diff: 0.0000
User: 1 | Item: 150 | KNN estimate: 1.8002 | UserBased estimate: 1.8002 | Diff: 0.0000
User: 1 | Item: 15

# 5. Compare MSD and Jacard
Compare predictions made with MSD similarity and Jacard similarity


In [14]:
# -- compare predictions made with MSD similarity and Jacard similarity --
